In [1]:
from dotenv import load_dotenv
import os
import clickhouse_connect
import pandas as pd
from datetime import datetime


load_dotenv()

client = clickhouse_connect.get_client(
    host=os.getenv('CLICKHOUSE_HOST'),
    port=int(os.getenv('CLICKHOUSE_PORT', 8123)), 
    username=os.getenv('CLICKHOUSE_USERNAME'),
    password=os.getenv('CLICKHOUSE_PASSWORD'),
    database=os.getenv('CLICKHOUSE_DATABASE')
)

In [2]:
# shipment_dataset_query = """ 
# SELECT 
#     bni.quantity_in,

#     -- From bills
#     b.bill_date,
#     b.due_date,
#     b.bill_status,
#     b.eta,
#     b.bill_type,
#     b.seal,
#     b.fda_status,
#     b.coo,
#     b.customs_broker,
#     b.receipt_date,
#     b.inventory,
#     b.gross_weight_lb,
#     b.ocean_freight,
#     b.transshipment_date_ett,
#     b.tariff_amount,
#     b.tariff_type,
#     b.place_of_delivery,
#     b.shipping_port,
#     b.consignee,

#     -- From items
#     i.purchase_price,
#     i.sales_price,
#     i.opening_stock,
#     i.cases AS item_cases,
#     i.product_type,
#     i.product_category,
#     i.brand,
#     i.manufacturer,
#     i.sales_account,
#     i.size AS item_size,
#     i.item_type,
#     i.cooking_style,
#     i.soaking_style,
#     i.yield_percentage,
#     i.status

# FROM 
#     zoho_books_analytics.batch_number_in bni
# JOIN 
#     zoho_books_analytics.bills b 
#     ON bni.bill_id = b.bill_id

# JOIN 
#     zoho_books_analytics.items i 
#     ON bni.product_id = i.item_id

# ORDER BY 
#     bni.created_time DESC

# """


# shipment_dataset_query = """ 
# SELECT 
#     bni.quantity_in,

#     -- From bills
#     b.bill_date,
#     b.shipped_date,
#     b.due_date,
#     b.eta AS eta,
#     b.seal,
#     b.customs_broker,
#     b.receipt_date,
#     b.gross_weight_lb,
#     b.ocean_freight,
#     i.yield_percentage

# FROM 
#     zoho_books_analytics.batch_number_in bni
# JOIN 
#     zoho_books_analytics.bills b 
#     ON bni.bill_id = b.bill_id
# JOIN 
#     zoho_books_analytics.bill_item bi 
#     ON bi.bill_id = b.bill_id
# JOIN 
#     zoho_books_analytics.purchase_orders po  
#     ON po.purchase_order_number = b.purchase_order
# JOIN 
#     zoho_books_analytics.items i 
#     ON i.item_id = bi.product_id
# JOIN 
#     zoho_books_analytics.customer_item_mapping cim 
#     ON i.sku = cim.az_sku

# WHERE 
#     cim.customer_name LIKE 'Walmart%'
#     AND po.po_commited != 'Direct Sale'

# ORDER BY 
#     bni.created_time DESC
# """


In [3]:
# shipment_dataset_query = """ 
# SELECT 
#     bni.quantity_in,
#     bni.created_time as batch_created_time,

#     -- From bills
#     b.bill_date,
#     b.coo,
#     b.shipping_port,
#     b.shipped_date,
#     b.due_date,
#     b.eta AS eta,
#     b.customs_broker,
#     b.receipt_date
# FROM zoho_books_analytics.batch_number_in bni
# INNER JOIN zoho_books_analytics.bills b on bni.bill_id = b.bill_id 
# INNER JOIN zoho_books_analytics.bill_item bi ON b.bill_id  = bi.bill_id
# INNER JOIN zoho_books_analytics.purchase_orders p ON b.purchase_order   = p.purchase_order_number  
# INNER JOIN zoho_books_analytics.items i ON bi.product_id   = i.item_id 
# INNER JOIN zoho_books_analytics.sales_orders so ON p.reference_number = so.sales_order 
# INNER JOIN zoho_books_analytics.customers c ON c.customer_id   = so.customer_id
# INNER JOIN zoho_books_analytics.customer_item_mapping ci ON i.sku   = ci.az_sku 
# INNER JOIN zoho_books_analytics.vendors v on v.vendor_id = b.vendor_id
# WHERE	 c.customer_name   like 'Walmart%'
# ORDER BY 
#     bni.created_time DESC
# """


In [4]:
shipment_dataset_query = """SELECT 
    bni.quantity_in,
    bni.created_time as batch_created_time,
    -- From bills
    b.bill_date,
    b.coo,
    b.shipping_port,
    b.shipped_date,
    b.due_date,
    b.eta AS eta,
    b.customs_broker,
    b.receipt_date,
    b.payment_terms,
    b.scac,
    b.tariff_amount,
    b.bill_status,
    i.sku,
    i.brand,
    i.manufacturer
FROM zoho_books_analytics.batch_number_in bni
INNER JOIN zoho_books_analytics.bills b on bni.bill_id = b.bill_id 
INNER JOIN zoho_books_analytics.bill_item bi ON b.bill_id  = bi.bill_id
INNER JOIN zoho_books_analytics.purchase_orders p ON b.purchase_order   = p.purchase_order_number  
INNER JOIN zoho_books_analytics.items i ON bi.product_id   = i.item_id 
INNER JOIN zoho_books_analytics.sales_orders so ON p.reference_number = so.sales_order 
INNER JOIN zoho_books_analytics.customers c ON c.customer_id   = so.customer_id
INNER JOIN zoho_books_analytics.customer_item_mapping ci ON i.sku   = ci.az_sku 
INNER JOIN zoho_books_analytics.vendors v on v.vendor_id = b.vendor_id
WHERE	 c.customer_name   like 'Walmart%'
ORDER BY 
    bni.created_time DESC"""

In [5]:
result = client.query(shipment_dataset_query)

shipment_dataset_df = pd.DataFrame(result.result_rows, columns=[col for col in result.column_names])

In [6]:
shipment_dataset_df.columns

Index(['quantity_in', 'batch_created_time', 'bill_date', 'b.coo',
       'shipping_port', 'shipped_date', 'due_date', 'eta', 'customs_broker',
       'receipt_date', 'b.payment_terms', 'scac', 'tariff_amount',
       'bill_status', 'sku', 'brand', 'manufacturer'],
      dtype='object')

In [7]:
date_time_columns = ['eta','receipt_date','due_date','bill_date','shipped_date', 'batch_created_time']
for col in date_time_columns:
    shipment_dataset_df[f'{col}'] = pd.to_datetime(shipment_dataset_df[f'{col}'], format='%d %b %Y', errors='raise')

# shipment_dataset_df['eta'] = pd.to_datetime(shipment_dataset_df['eta'], format='%d %b %Y', errors='raise')
# shipment_dataset_df['receipt_date'] = pd.to_datetime(shipment_dataset_df['receipt_date'], format='%d %b %Y', errors='raise')
# shipment_dataset_df['due_date'] = pd.to_datetime(shipment_dataset_df['due_date'], format='%d %b %Y', errors='raise')
# shipment_dataset_df['bill_date'] = pd.to_datetime(shipment_dataset_df['bill_date'], format='%d %b %Y', errors='raise')
# # shipment_dataset_df['transshipment_date_ett'] = pd.to_datetime(shipment_dataset_df['transshipment_date_ett'], format='%d %b %Y', errors='raise')
# shipment_dataset_df['shipped_date'] = pd.to_datetime(shipment_dataset_df['shipped_date'], format='%d %b %Y', errors='raise')


# # Replace NaT (null) with today's date
# shipment_dataset_df['receipt_date'].fillna(pd.Timestamp(datetime.today().strftime('%Y-%m-%d')), inplace=True)

In [8]:
shipment_dataset_df.dropna(subset='receipt_date', inplace=True)

In [9]:
# Compute shipment delay in days
shipment_dataset_df['shipment_delay_days'] = (
    shipment_dataset_df['receipt_date'] - shipment_dataset_df['eta']
).dt.days

In [10]:
THRESHOLD = 7

In [11]:
shipment_dataset_df['shipment_classified'] = shipment_dataset_df['shipment_delay_days'].apply(
    lambda x: 'on_time' if x <= THRESHOLD else 'delayed')

In [12]:
shipment_dataset_df.to_csv('./results/raw_shipment_classification_dataset.csv', index=False)